In [ ]:
!pip install -q langchain-openai langgraph pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 27.6 MB/s eta 0:00:00


In [ ]:
!pip install -q -U langchain-openai langchain-core langgraph pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 71.7 MB/s eta 0:00:00


In [8]:
!pip install -q -U langchain-openai langchain-core langgraph pydantic

import re
from typing import Annotated, Literal, TypedDict

from google.colab import userdata

from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage
)

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages

from pydantic import BaseModel, Field

In [10]:
!pip install -q -U langchain-openai langchain-core langgraph pydantic

In [12]:
import re
from typing import Annotated, Literal, TypedDict
from google.colab import userdata
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage
  )
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages

from pydantic import BaseModel, Field

In [14]:
try:
    openrouter_key = userdata.get("GeminiAPIKey4")

    if not openrouter_key:
        raise ValueError("API key is empty.")

    print("API key loaded successfully.")
    print("Secret name: GeminiAPIKey4")

except Exception as e:
    raise RuntimeError(
        "\nAPI key could not be loaded.\n\n"
        "Go to Google Colab -> Secrets -> Add new secret\n"
        "and create:\n\n"
        "Name: GeminiAPIKey4\n"
        "Value: Your OpenRouter API key\n\n"
        "Also enable Notebook access."
    ) from e


API key loaded successfully.
Secret name: GeminiAPIKey4


In [15]:
SELECTED_MODEL = "google/gemini-2.5-flash"

print("--------------------------------------------")
print("LangGraph Ticket Router")
print("--------------------------------------------")
print("Model:", SELECTED_MODEL)
print("Provider: OpenRouter")

--------------------------------------------
LangGraph Ticket Router
--------------------------------------------
Model: google/gemini-2.5-flash
Provider: OpenRouter


In [16]:
llm = ChatOpenAI(
    model=SELECTED_MODEL,
    api_key=openrouter_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

print("LLM initialized successfully.")

LLM initialized successfully.


In [17]:
def extract_text_safely(content) -> str:

    if isinstance(content, str):
        return content

    if isinstance(content, list):

        parts = []

        for item in content:

            if isinstance(item, dict):

                if "text" in item:
                    parts.append(str(item["text"]))

                elif "content" in item:
                    parts.append(str(item["content"]))

                else:
                    parts.append(str(item))

            elif isinstance(item, str):
                parts.append(item)

            else:
                parts.append(str(item))

        return "".join(parts)

    return str(content)

In [18]:
class AgentState(TypedDict):

    messages: Annotated[
        list[BaseMessage],
        add_messages
    ]

    next_node: str

In [19]:
class RouteDecision(BaseModel):

    destination: Literal[
        "tech_support_agent",
        "account_actions_agent"
    ] = Field(
        description=(
            "Choose tech_support_agent for technical problems, "
            "bugs, application problems, software problems, "
            "device problems and technical questions. "
            "Choose account_actions_agent for refunds, billing, "
            "payments and account actions."
        )
    )

In [20]:
@tool
def process_refund(user_id: str, amount: float) -> str:
    """
    Processes a refund for a user.
    """

    return (
        f"SUCCESS: Refund of ${amount:.2f} "
        f"has been processed for User '{user_id}'."
    )

In [21]:
def supervisor_agent(state: AgentState) -> dict:

    system_prompt = SystemMessage(
        content=(
            "You are an intent routing classifier.\n\n"

            "Classify the user's request into exactly ONE category.\n\n"

            "TECH means:\n"
            "- Technical questions\n"
            "- Troubleshooting\n"
            "- Software problems\n"
            "- Device problems\n"
            "- Bugs\n"
            "- Errors\n"
            "- Application problems\n"
            "- Technical how-to questions\n\n"

            "ACCOUNT means:\n"
            "- Refunds\n"
            "- Billing\n"
            "- Payments\n"
            "- Account actions\n"
            "- Account-related requests\n\n"

            "Reply with ONLY one word:\n"
            "TECH\n"
            "or\n"
            "ACCOUNT"
        )
    )

    messages = [
        system_prompt
    ] + state["messages"]

    try:

        response = llm.invoke(messages)

        text = extract_text_safely(
            response.content
        ).strip().upper()

        print("\nSupervisor response:", text)

        if "ACCOUNT" in text:
            destination = "account_actions_agent"
        else:
            destination = "tech_support_agent"

        print(
            "Routing to:",
            destination
        )

        return {
            "next_node": destination
        }

    except Exception as e:

        print(
            "\nSupervisor error:",
            type(e).__name__,
            str(e)
        )

        return {
            "next_node": "tech_support_agent"
        }

In [22]:
def tech_support_agent(state: AgentState) -> dict:

    system_prompt = SystemMessage(
        content=(
            "You are a Technical Support Specialist.\n\n"
            "Give short, clear and useful answers.\n"
            "Use simple language.\n"
            "Provide step-by-step instructions when appropriate."
        )
    )

    messages = [
        system_prompt
    ] + state["messages"]

    response = llm.invoke(messages)

    text_content = extract_text_safely(
        response.content
    )

    final_message = (
        "[Tech Support Agent]\n\n"
        f"{text_content}"
    )

    return {
        "messages": [
            AIMessage(
                content=final_message
            )
        ],
        "next_node": END
    }

In [23]:
def account_actions_agent(state: AgentState) -> dict:

    user_text = ""

    for message in reversed(state["messages"]):

        if isinstance(message, HumanMessage):

            user_text = extract_text_safely(
                message.content
            )

            break

    # ========================================================
    # REFUND
    # ========================================================

    if "refund" in user_text.lower():

        # ----------------------------------------------------
        # Extract amount
        # ----------------------------------------------------

        amount_match = re.search(
            r"\$\s*(\d+(?:\.\d+)?)",
            user_text
        )

        if amount_match:

            amount = float(
                amount_match.group(1)
            )

        else:

            amount_match = re.search(
                r"(\d+(?:\.\d+)?)\s*(?:dollars|usd)",
                user_text,
                re.IGNORECASE
            )

            if amount_match:

                amount = float(
                    amount_match.group(1)
                )

            else:

                amount = None

        # ----------------------------------------------------
        # Extract User ID
        # ----------------------------------------------------

        user_id_match = re.search(
            r"(?:user\s*id\s*[:\-]?\s*)([A-Za-z0-9_-]+)",
            user_text,
            re.IGNORECASE
        )

        if user_id_match:

            user_id = user_id_match.group(1)

        else:

            user_id_match = re.search(
                r"\bUSER[A-Za-z0-9_-]+\b",
                user_text,
                re.IGNORECASE
            )

            if user_id_match:

                user_id = user_id_match.group(0)

            else:

                user_id = None

        # ----------------------------------------------------
        # PROCESS REFUND
        # ----------------------------------------------------

        if (
            amount is not None
            and
            user_id is not None
        ):

            tool_output = process_refund.invoke(
                {
                    "user_id": user_id,
                    "amount": amount
                }
            )

            final_message = (
                "[Account Actions Agent]\n\n"
                "Tool Execution:\n"
                f"{tool_output}"
            )

        else:

            missing = []

            if amount is None:

                missing.append(
                    "refund amount"
                )

            if user_id is None:

                missing.append(
                    "user ID"
                )

            final_message = (
                "[Account Actions Agent]\n\n"
                "I need the following information "
                "to process the refund:\n"
                f"- {', '.join(missing)}"
            )

    # ========================================================
    # OTHER ACCOUNT REQUEST
    # ========================================================

    else:

        system_prompt = SystemMessage(
            content=(
                "You are an Account Manager.\n"
                "Answer billing, payment and account "
                "questions clearly and simply."
            )
        )

        messages = [
            system_prompt
        ] + state["messages"]

        response = llm.invoke(
            messages
        )

        text_content = extract_text_safely(
            response.content
        )

        final_message = (
            "[Account Actions Agent]\n\n"
            f"{text_content}"
        )

    return {
        "messages": [
            AIMessage(
                content=final_message
            )
        ],
        "next_node": END
    }

In [24]:
workflow = StateGraph(
    AgentState
)

# ============================================================
# ADD NODES
# ============================================================

workflow.add_node(
    "supervisor",
    supervisor_agent
)

workflow.add_node(
    "tech_support_agent",
    tech_support_agent
)

workflow.add_node(
    "account_actions_agent",
    account_actions_agent
)

# ============================================================
# START -> SUPERVISOR
# ============================================================

workflow.add_edge(
    START,
    "supervisor"
)

# ============================================================
# SUPERVISOR -> AGENT
# ============================================================

workflow.add_conditional_edges(
    "supervisor",

    lambda state:
        state["next_node"],

    {
        "tech_support_agent":
            "tech_support_agent",

        "account_actions_agent":
            "account_actions_agent"
    }
)

# ============================================================
# AGENTS -> END
# ============================================================

workflow.add_edge(
    "tech_support_agent",
    END
)

workflow.add_edge(
    "account_actions_agent",
    END
)

In [25]:
app = workflow.compile()

print()
print("============================================")
print("LangGraph workflow compiled successfully!")
print("============================================")


LangGraph workflow compiled successfully!


In [26]:
def run_demo(user_query: str):

    print()
    print("=" * 60)
    print("USER QUERY")
    print("=" * 60)

    print(user_query)

    inputs = {
        "messages": [
            HumanMessage(
                content=user_query
            )
        ],
        "next_node": ""
    }

    try:

        result = app.invoke(
            inputs
        )

        print()
        print("=" * 60)
        print("SYSTEM RESPONSE")
        print("=" * 60)

        print(
            result["messages"][-1].content
        )

    except Exception as e:

        print()
        print("=" * 60)
        print("ERROR")
        print("=" * 60)

        print(
            type(e).__name__
        )

        print(
            str(e)
        )

In [27]:
run_demo(
    "How can I learn using katana?"
)


USER QUERY
How can I learn using katana?

Supervisor error: OpenAIAuthenticationError Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}

ERROR
OpenAIAuthenticationError
Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}


In [28]:
run_demo(
    "My application is freezing and crashing. "
    "How can I fix it?"
)


USER QUERY
My application is freezing and crashing. How can I fix it?

Supervisor error: OpenAIAuthenticationError Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}

ERROR
OpenAIAuthenticationError
Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}


In [29]:
run_demo(
    "Please refund $50 for user ID USER123."
)


USER QUERY
Please refund $50 for user ID USER123.

Supervisor error: OpenAIAuthenticationError Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}

ERROR
OpenAIAuthenticationError
Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}
